# Paso 10 — Mapa de México coloreado con k1coloring desde Neo4j

## 1. Instalación de dependencias

In [ ]:
# !pip install neo4j geopandas matplotlib

## 2. Imports

In [15]:
import os
import unicodedata
import urllib.request
import matplotlib
matplotlib.use('TkAgg')   # cambiar a 'Agg' si no hay pantalla disponible
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
from neo4j import GraphDatabase

## 3. Conexión a Neo4j y lectura de k1color

In [ ]:
URI      = "bolt://localhost:7687"
USER     = "neo4j"
PASSWORD = ""
DATABASE = "mex"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

with driver.session(database=DATABASE) as session:
    result = session.run("MATCH (e:Estado) RETURN e.nombre AS nombre, e.k1color AS color")
    records = [(r["nombre"], r["color"]) for r in result]

driver.close()

coloring = {nombre: color for nombre, color in records}
print(f"✅ {len(coloring)} estados cargados desde Neo4j")
for nombre, color in sorted(coloring.items(), key=lambda x: x[1]):
    print(f"   {color}  {nombre}")

✅ 32 estados cargados desde Neo4j
   1  BAJA CALIFORNIA SUR
   1  CHIHUAHUA
   1  HIDALGO
   1  JALISCO
   1  MORELOS
   1  NUEVO LEÓN
   1  OAXACA
   1  QUINTANA ROO
   1  TABASCO
   2  DURANGO
   2  MÉXICO
   2  SAN LUIS POTOSÍ
   2  SONORA
   2  YUCATÁN
   3  MICHOACÁN
   3  PUEBLA
   3  QUERÉTARO
   3  SINALOA
   4  VERACRUZ
   4  ZACATECAS
   5  AGUASCALIENTES
   5  BAJA CALIFORNIA
   5  CAMPECHE
   5  COAHUILA
   5  COLIMA
   5  CHIAPAS
   5  CIUDAD DE MÉXICO
   5  GUANAJUATO
   5  GUERRERO
   5  NAYARIT
   5  TAMAULIPAS
   5  TLAXCALA


## 4. Descargar GeoJSON de estados de México

In [19]:
GEOJSON_PATH = "mexico_estados.geojson"
GEOJSON_URL  = "https://raw.githubusercontent.com/angelnmara/geojson/master/mexicoHigh.json"

if not os.path.exists(GEOJSON_PATH):
    print(f"Descargando shapefile desde:\n{GEOJSON_URL}")
    urllib.request.urlretrieve(GEOJSON_URL, GEOJSON_PATH)
    print("✅ Descargado")
else:
    print("✅ GeoJSON ya existe, no se descarga de nuevo")

gdf = gpd.read_file(GEOJSON_PATH)
print(f"\n{len(gdf)} polígonos en el GeoDataFrame")
print("Columnas disponibles:", gdf.columns.tolist())
print("\nPrimeras 5 filas de todas las columnas de texto:")
print(gdf.drop(columns="geometry").head(5).to_string())

✅ GeoJSON ya existe, no se descarga de nuevo

32 polígonos en el GeoDataFrame
Columnas disponibles: ['name', 'id', 'CNTRY', 'TYPE', 'geometry']

Primeras 5 filas de todas las columnas de texto:
         name      id   CNTRY   TYPE
0   Zacatecas  MX-ZAC  Mexico  State
1     Yucatán  MX-YUC  Mexico  State
2    Veracruz  MX-VER  Mexico  State
3    Tlaxcala  MX-TLA  Mexico  State
4  Tamaulipas  MX-TAM  Mexico  State


## 5. Detectar columna de nombre y normalizar para el join

El script busca automáticamente qué columna del GeoJSON
contiene los nombres de los estados, normalizando tildes y
mayúsculas para hacer el join aunque los nombres no sean idénticos.

In [20]:
def normalizar(texto):
    """Quita tildes, pasa a mayúsculas y elimina espacios extra."""
    if not isinstance(texto, str):
        return ""
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return texto.upper().strip()

# Nombres Neo4j normalizados → nombre Neo4j original
neo4j_norm = {normalizar(k): k for k in coloring.keys()}

# Buscar la columna del GeoJSON que mejor hace match con los estados
text_cols = [c for c in gdf.columns if c != "geometry" and gdf[c].dtype == object]
best_col, best_hits = None, 0

for col in text_cols:
    hits = gdf[col].apply(lambda v: normalizar(v) in neo4j_norm).sum()
    print(f"  Columna '{col}': {hits}/{len(gdf)} coincidencias")
    if hits > best_hits:
        best_hits, best_col = hits, col

print(f"\n✅ Columna seleccionada: '{best_col}' ({best_hits}/{len(gdf)} coincidencias)")

# Mapear cada fila del GeoDataFrame a su nombre en Neo4j y su k1color
gdf["neo4j_nombre"] = gdf[best_col].apply(lambda v: neo4j_norm.get(normalizar(v)))
gdf["k1color"]      = gdf["neo4j_nombre"].map(coloring)

sin_color = gdf[gdf["k1color"].isna()]
if len(sin_color) > 0:
    print("\n⚠️  Filas sin color asignado:")
    print(sin_color[[best_col, "neo4j_nombre"]].to_string())
else:
    print("✅ Todos los estados tienen color asignado")

  Columna 'name': 32/32 coincidencias
  Columna 'id': 0/32 coincidencias
  Columna 'CNTRY': 32/32 coincidencias
  Columna 'TYPE': 0/32 coincidencias

✅ Columna seleccionada: 'name' (32/32 coincidencias)
✅ Todos los estados tienen color asignado


## 6. Paleta de colores

In [21]:
paleta = {
    1: "#E63946",   # Rojo
    2: "#2A9D8F",   # Verde Teal
    3: "#E9C46A",   # Amarillo
    4: "#6A4C93",   # Morado
    5: "#4A90D9",   # Azul  (antes color 0, reasignado en el Paso 5 del CQL)
}
nombre_color = {
    1: "Color 1 – Rojo",
    2: "Color 2 – Verde",
    3: "Color 3 – Amarillo",
    4: "Color 4 – Morado",
    5: "Color 5 – Azul",
}

gdf["hex_color"] = gdf["k1color"].map(paleta).fillna("#cccccc")

print("Distribución de colores asignados:")
print(gdf.groupby("k1color")["neo4j_nombre"].apply(list).to_string())

Distribución de colores asignados:
k1color
1    [TABASCO, QUINTANA ROO, OAXACA, NUEVO LEÓN, MO...
2    [YUCATÁN, SONORA, SAN LUIS POTOSÍ, MÉXICO, DUR...
3              [SINALOA, QUERÉTARO, PUEBLA, MICHOACÁN]
4                                [ZACATECAS, VERACRUZ]
5    [TLAXCALA, TAMAULIPAS, NAYARIT, GUERRERO, GUAN...


## 7. Dibujar el mapa

In [22]:
abrev = {
    "AGUASCALIENTES": "AGS",    "BAJA CALIFORNIA": "BC",       "BAJA CALIFORNIA SUR": "BCS",
    "CAMPECHE": "CAM",          "CHIAPAS": "CHIS",              "CHIHUAHUA": "CHIH",
    "CIUDAD DE MEXICO": "CDMX", "COAHUILA": "COAH",             "COLIMA": "COL",
    "DURANGO": "DGO",           "GUANAJUATO": "GTO",            "GUERRERO": "GRO",
    "HIDALGO": "HGO",           "JALISCO": "JAL",               "MEXICO": "MEX",
    "MICHOACAN": "MICH",        "MORELOS": "MOR",               "NAYARIT": "NAY",
    "NUEVO LEON": "NL",         "OAXACA": "OAX",                "PUEBLA": "PUE",
    "QUERETARO": "QRO",         "QUINTANA ROO": "QROO",         "SAN LUIS POTOSI": "SLP",
    "SINALOA": "SIN",           "SONORA": "SON",                "TABASCO": "TAB",
    "TAMAULIPAS": "TAMPS",      "TLAXCALA": "TLAX",             "VERACRUZ": "VER",
    "YUCATAN": "YUC",           "ZACATECAS": "ZAC",
}

fig, ax = plt.subplots(figsize=(16, 12))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#0d2137")

gdf.plot(
    ax=ax,
    color=gdf["hex_color"],
    edgecolor="white",
    linewidth=0.6,
)

for _, row in gdf.iterrows():
    if row.geometry is None:
        continue
    try:
        centroid  = row.geometry.centroid
        # usamos el nombre normalizado (sin tildes) para buscar la abreviatura
        nombre_norm = normalizar(row.get("neo4j_nombre", "") or "")
        label     = abrev.get(nombre_norm, "")
        txt_color = "#1a1a2e" if row.get("k1color") == 3 else "white"
        ax.annotate(
            label,
            xy=(centroid.x, centroid.y),
            ha="center", va="center",
            fontsize=6.5, fontweight="bold",
            color=txt_color,
        )
    except Exception:
        pass

patches = [
    mpatches.Patch(facecolor=paleta[c], edgecolor="white", linewidth=0.8, label=nombre_color[c])
    for c in sorted(paleta)
]
legend = ax.legend(
    handles=patches,
    loc="lower left",
    framealpha=0.85,
    facecolor="#0d2137",
    edgecolor="#444",
    labelcolor="white",
    fontsize=9,
    title="k1color  (Neo4j GDS)",
    title_fontsize=9.5,
)
legend.get_title().set_color("white")

ax.set_title(
    "República Mexicana — K1 Coloring\nNeo4j GDS · Base de datos: mex · Instancia: MdG",
    fontsize=14, fontweight="bold", color="white", pad=12,
)
ax.axis("off")
plt.tight_layout()

## 8. Guardar y mostrar

In [23]:
plt.savefig("mapa_k1coloring.png", dpi=180, bbox_inches="tight",
            facecolor="#1a1a2e", edgecolor="none")
print("✅ Mapa guardado como mapa_k1coloring.png")
plt.show()

✅ Mapa guardado como mapa_k1coloring.png
